# Lichess: reproducible collection pilot

Collect rated standard **1+0, 5+0, 10+0** games with complete legal mainlines and clocks. The collector uses seeded hash priorities and disk storage. A full-month sample reads the **entire archive**, even after quotas fill.

The existing 90k games are all from October 1; preserve them as a pilot. Lecturer approval and the Kaggle source check remain prerequisites for the substantial full-month collection. No models or Stockfish labels are produced here.


## 1. Environment

Once, from this folder: `.venv/bin/python -m pip install -r requirements.txt`. Select the project environment as your notebook kernel. The script and notebook should be in the same folder.


In [ ]:
import sys, json, csv, subprocess, importlib.util
from pathlib import Path
from collections import Counter

PROJECT = Path.cwd()
SCRIPT = PROJECT / "02_lichess_stratified_sample.py"
assert SCRIPT.is_file(), "Open this notebook from the sampling project folder."
missing = [p for p in ("chess", "requests", "zstandard") if importlib.util.find_spec(p) is None]
assert not missing, f"Install requirements in this kernel; missing: {missing}"
print("Python:", sys.executable)
print("Project:", PROJECT)


## 2. Small pilot settings

The verified live test is in `data/live_pilot_100_per_control`. Set `RUN_COLLECTION=True` and choose a **new** output directory to run another test. The collector refuses to overwrite existing data.

A 100,000-record cap tests the pipeline on a month prefix. It does **not** establish full-month sampling. The output explicitly records this limitation.


In [ ]:
MONTH = "2026-10"
SEED = 2026
QUOTA = 100
MAX_GAMES = 100_000
OUTPUT = PROJECT / "data" / "live_pilot_100_per_control"
LOCAL_ARCHIVE = None  # Or Path("lichess_db_standard_rated_2026-10.pgn.zst")
RUN_COLLECTION = False

command = [sys.executable, str(SCRIPT), "--month", MONTH, "--seed", str(SEED),
           "--quota", str(QUOTA), "--max-games", str(MAX_GAMES), "--output", str(OUTPUT)]
if LOCAL_ARCHIVE is not None:
    command += ["--local", str(LOCAL_ARCHIVE)]
print(" ".join(command))


In [ ]:
if RUN_COLLECTION:
    subprocess.run(command, check=True)  # Progress appears while reading.
else:
    print("Collection disabled; inspecting the saved pilot below.")


## 3. Inspect and independently validate

Check quotas, date coverage, source settings and completion status. Then replay the saved PGNs and compare their IDs, fields, counts and hashes with the CSV and manifest.


In [ ]:
manifest = json.loads((OUTPUT / "dataset_manifest.json").read_text())
print(json.dumps({k: manifest[k] for k in ["status", "complete", "whole_source_read",
      "source_integrity_verified", "counts", "statistics", "elapsed_seconds"]}, indent=2))
with (OUTPUT / "games_metadata.csv").open(newline="") as f:
    rows = list(csv.DictReader(f))
print("Dates by control:", Counter((r["time_control"], r["utc_date"]) for r in rows))
print("Unique IDs:", len({r["game_id"] for r in rows}), "of", len(rows))
print("Example metadata:", rows[0] if rows else None)


In [ ]:
subprocess.run([sys.executable, str(SCRIPT), "--validate-only", "--output", str(OUTPUT)], check=True)
report = json.loads((OUTPUT / "validation_report.json").read_text())
assert report["valid"]
print("Independent validation passed.")


## 4. Scale only after lecturer approval

Use `QUOTA=30_000`, `MAX_GAMES=0`, and a new output folder. Prefer a resumably downloaded local `.pgn.zst` (~31 GB); see the README for download and checksum commands. Remote streaming also works, but interrupted streams restart from the beginning. The full pass may take hours and has **not** been run here.

The final manifest must report `complete=True`; inspect selected dates across the month and run independent validation. Sampling heaps stay in memory; selected raw PGNs remain on disk. Do not decompress the entire month to a huge plain PGN.

Later, `%clk` on a move is its **post-move** clock. Pre-move time normally comes from that player's previous move; handle first moves and suspected berserk separately. Split entire games before extracting modeling rows. Benchmark local Stockfish on a small position subset before choosing labeling volume.
